# IDR Prediction — Augmented Training
Trains Transformer-LSTM on real DisProt + GAN-generated synthetic data (1:1 ratio).
Compares CAID F1/AUC against the baseline trained on real data only.

**Prerequisites:** run `train_gan.ipynb` first to generate `synthetic_disprot.json`.
**Run all cells top to bottom.**

In [ ]:
# ── 1. Environment setup (Colab only) ────────────────────────────────────────
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone https://github.com/siavashprh/idr-prediction.git
    os.chdir('idr-prediction')
    !pip install -q -e .
    !pip install -q 'transformers==5.7.0' 'tokenizers==0.22.2' biopython omegaconf sentencepiece safetensors
    import torch
    if tuple(int(x) for x in torch.__version__.split('.')[:2]) < (2, 6):
        !pip install -q 'torch>=2.6.0'

    from google.colab import files
    print('Upload synthetic_disprot.json when prompted:')
    uploaded = files.upload()
    os.makedirs('data_repository/processed', exist_ok=True)
    for fname in uploaded:
        os.rename(fname, f'data_repository/processed/{fname}')

print('Working dir:', os.getcwd())

In [ ]:
# ── 2. Imports ───────────────────────────────────────────────────────────────
import json, random, time
from IPython.display import clear_output

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import transformers
from tqdm.notebook import tqdm
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score

transformers.logging.set_verbosity_error()

from src.config import ModelConfig
from src.data.data import IDPData, get_caid_data, get_disprot_cut_data
from src.models.sequence_models import TransformerLSTMModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── 3. Config ────────────────────────────────────────────────────────────────
from omegaconf import OmegaConf
cfg = OmegaConf.load('configs/augmented.yaml')
print(OmegaConf.to_yaml(cfg))

In [ ]:
# ── 4. Data ──────────────────────────────────────────────────────────────────
print('Loading real training data...')
real = get_disprot_cut_data()

print('Loading synthetic training data...')
with open(cfg.data.synthetic_path) as f:
    synthetic = json.load(f)

all_seqs   = real['sequences']   + synthetic['sequences']
all_labels = real['disorder region'] + synthetic['disorder region']
train_data = IDPData(all_seqs, all_labels)

print(f'  Real chunks      : {len(real["sequences"])}')
print(f'  Synthetic chunks : {len(synthetic["sequences"])}')
print(f'  Total            : {len(train_data.X)}')

print('Loading CAID data...')
caid = get_caid_data()
print(f'  CAID sequences   : {len(caid["sequences"])}')

In [ ]:
# ── 5. Model ─────────────────────────────────────────────────────────────────
print('Loading prot_bert_bfd...')
model_config = ModelConfig()
model_config.model_name = cfg.model.name

model = TransformerLSTMModel(
    pre_model_name   = cfg.model.pre_model,
    device           = device,
    input_dim        = cfg.model.input_dim,
    linear_hidden_dim= cfg.model.linear_hidden_dim,
    num_heads        = cfg.model.num_heads,
    num_blocks       = cfg.model.num_blocks,
    dropout          = cfg.model.dropout,
    model_config     = model_config,
    with_lstm        = cfg.model.with_lstm,
    lstm_n_layers    = cfg.model.lstm_n_layers,
)
print('Model ready.')

In [ ]:
# ── 6. Training loop ─────────────────────────────────────────────────────────
loss_fn = nn.CrossEntropyLoss(
    ignore_index=2,
    weight=torch.tensor([cfg.training.first_class_weight,
                         1 - cfg.training.first_class_weight]).to(device),
)
optimizer = torch.optim.Adam(model.model.parameters(), lr=cfg.training.lr)

batch_size  = cfg.training.batch_size
n_epoch     = cfg.training.n_epoch
n           = len(train_data.X)
round_size  = n - (n % batch_size) - batch_size

epoch_losses, all_batch_losses = [], []
t_start = time.time()

for epoch in range(n_epoch):
    model.model.train()
    shuffle = list(range(n))
    random.shuffle(shuffle)
    batch_losses = []
    pbar = tqdm(range(0, round_size, batch_size), desc=f'Epoch {epoch+1}/{n_epoch}', leave=True)

    for j in pbar:
        indices = shuffle[j : j + batch_size]
        X = [train_data.X[i] for i in indices]
        max_len = max(len(s) for s in X)

        y = torch.full((batch_size, max_len), 2).long().to(device)
        for k, i in enumerate(indices):
            reg = train_data.y[i]
            y[k, :len(reg)] = torch.tensor(reg, dtype=torch.long)

        pred = model.model(X)
        loss = loss_fn(pred.reshape(-1, 2), y.reshape(-1))
        optimizer.zero_grad(); loss.backward(); optimizer.step()

        batch_losses.append(loss.item())
        all_batch_losses.append(loss.item())
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'avg': f'{np.mean(batch_losses):.4f}'})

    epoch_losses.append(np.mean(batch_losses))
    elapsed = (time.time() - t_start) / 60

    clear_output(wait=True)
    print(f'Epoch {epoch+1}: mean_loss={epoch_losses[-1]:.4f}  ({elapsed:.1f} min elapsed)')

    steps_per_epoch = round_size // batch_size
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(all_batch_losses, alpha=0.4, color='steelblue')
    for e in range(1, epoch + 2):
        ax.axvline(e * steps_per_epoch, color='gray', linestyle='--', linewidth=0.8)
    ax.set_xlabel('batch'); ax.set_ylabel('loss')
    ax.set_title(f'Training loss — epoch {epoch+1}/{n_epoch}  |  {elapsed:.1f} min')
    plt.tight_layout(); plt.show(); plt.close()

print(f'\nTraining complete in {(time.time()-t_start)/60:.1f} min')

In [ ]:
# ── 7. Save checkpoint ───────────────────────────────────────────────────────
os.makedirs('data_repository/ckpt', exist_ok=True)
ckpt_path = f'data_repository/ckpt/{cfg.model.name}.pth'
torch.save(model.model.state_dict(), ckpt_path)
print(f'Checkpoint saved: {ckpt_path}')

if IN_COLAB:
    from google.colab import files
    files.download(ckpt_path)
    print('Browser download triggered.')

In [ ]:
# ── 8. CAID evaluation & comparison ──────────────────────────────────────────
def predict_proba_sequence(model, sequence, max_length=400, alpha=0.75):
    if len(sequence) <= max_length:
        p = model.predict_proba([sequence])
        return np.array(p if isinstance(p, list) else [p])
    y = np.zeros(len(sequence))
    weight = np.zeros(len(sequence))
    start = 0
    while start < len(sequence):
        end = min(start + max_length, len(sequence))
        p = model.predict_proba([sequence[start:end]])
        chunk = np.array(p if isinstance(p, list) else [p])
        y[start:end] += chunk
        weight[start:end] += 1.0
        if end == len(sequence): break
        start += int(max_length * alpha)
    return y / np.maximum(weight, 1.0)

model.model.eval()
all_true, all_pred, all_proba = [], [], []

for seq, labels in tqdm(zip(caid['sequences'], caid['disorder region']),
                        total=len(caid['sequences']), desc='CAID eval'):
    with torch.no_grad():
        proba = predict_proba_sequence(model, list(seq))
    pred = (proba >= 0.5).astype(int)
    all_true.extend(labels)
    all_pred.extend(pred.tolist())
    all_proba.extend(proba.tolist())

all_true = np.array(all_true)
all_pred = np.array(all_pred)
all_proba = np.array(all_proba)

augmented_metrics = {
    'F1 (macro)' : f1_score(all_true, all_pred, average='macro', zero_division=1),
    'Precision'  : precision_score(all_true, all_pred, average='macro', zero_division=1),
    'Recall'     : recall_score(all_true, all_pred, average='macro', zero_division=1),
    'AUC-ROC'    : roc_auc_score(all_true, all_proba),
}

f1_per_class = f1_score(all_true, all_pred, average=None, zero_division=1)

# Baseline results from train_baseline.ipynb
baseline_metrics = {
    'F1 (macro)' : 0.5838,
    'Precision'  : 0.6087,
    'Recall'     : 0.6963,
    'AUC-ROC'    : 0.7381,
}

print('\n── CAID Results ──────────────────────────────')
print(f'{"Metric":<16} {"Baseline":>10} {"Augmented":>10} {"Delta":>8}')
print('-' * 46)
for k in baseline_metrics:
    b = baseline_metrics[k]
    a = augmented_metrics[k]
    print(f'{k:<16} {b:>10.4f} {a:>10.4f} {a-b:>+8.4f}')

print(f'\nPer-class F1:')
print(f'  Ordered class   : {f1_per_class[0]:.4f}')
print(f'  Disorder class  : {f1_per_class[1]:.4f}')
print(f'\nThesis baseline F1 target: ~0.618')